# 📊 Notebook 08: Advanced Categorical Plots

## 🎯 Learning Objectives

In this notebook, you'll master:

- **`barplot()`** with error bars and custom estimators
- When to use **`barplot()` vs `countplot()`**
- Custom aggregation functions (median, percentiles, custom)
- Grouped bar plots with `hue` parameter
- **Comprehensive comparison** of all categorical plot types
- Statistical annotations and significance markers
- Best practices for categorical data visualization

---

## 📚 Categorical Plot Types in Seaborn

| Plot Type | Use Case | Estimator | Shows Distribution |
|-----------|----------|-----------|-------------------|
| `countplot()` | Frequency counts | count | No |
| `barplot()` | Aggregated values with error bars | mean (customizable) | No |
| `boxplot()` | Quartiles and outliers | N/A | Yes |
| `violinplot()` | Distribution shape + quartiles | N/A | Yes |
| `stripplot()` | Individual points (jittered) | N/A | Yes |
| `swarmplot()` | Individual points (positioned) | N/A | Yes |
| `pointplot()` | Trends with error bars | mean (customizable) | No |

---

In [ ]:
# 📦 Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import sys
sys.path.append('../')
from utils.plot_utils import apply_theme, save_fig, stylize_plot

# Apply consistent theme
apply_theme()

In [ ]:
# 📊 Load datasets
tips = sns.load_dataset("tips")
titanic = sns.load_dataset("titanic")
employee_df = pd.read_csv("../datasets/employee_data.csv")

print("✅ Datasets loaded successfully")
print(f"Tips dataset shape: {tips.shape}")
print(f"Employee dataset shape: {employee_df.shape}")

## 1️⃣ `barplot()` - Aggregated Values with Error Bars

**`barplot()`** shows the **central tendency** (default: mean) of a numeric variable for each category, with error bars showing uncertainty.

**Key Parameters:**
- `estimator`: Function to aggregate (default: `np.mean`)
- `errorbar`: Type of error bars (`"ci"`, `"sd"`, `"se"`, `("pi", 95)`, or None)
- `hue`: Group by another categorical variable
- `order`: Control category order

**When to Use:**
- Comparing average values across categories
- Showing aggregated metrics with uncertainty
- Grouped comparisons with hue parameter

In [ ]:
# 📊 Basic barplot with default estimator (mean) and error bars
plt.figure(figsize=(10, 5 ))
sns.barplot(
    data=tips,
    x="day",
    y="total_bill",
    errorbar="ci",  # 👈 95% confidence interval (default)
    palette="Set2",
    alpha=0.8
)
stylize_plot(
    title="Average Total Bill by Day (with 95% CI)",
    xlabel="Day of Week",
    ylabel="Average Total Bill ($)"
)
save_fig("../exports/08_categorical/barplot_basic.png")
plt.show()

print("✅ Basic barplot with mean and confidence intervals")

## 2️⃣ `barplot()` vs `countplot()` - Key Differences

| Feature | `countplot()` | `barplot()` |
|---|---|---|
| **Purpose** | Count frequency of categories | Aggregate numeric values |
| **Y-axis** | Always count | Numeric variable (mean, median, etc.) |
| **Estimator** | Count only | Customizable (mean, median, sum, custom) |
| **Error Bars** | No | Yes (CI, SD, SE) |
| **Use When** | "How many in each category?" | "What's the average/total/median per category?" |

**Example Questions:**
- **countplot**: "How many transactions per day?"
- **barplot**: "What's the average transaction amount per day?"

In [ ]:
# 📊 Side-by-side comparison: barplot vs countplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# countplot: Frequency count
sns.countplot(data=tips, x="day", palette="Set3", ax=axes[0])
axes[0].set_title("countplot(): Count of Transactions by Day", fontweight="bold")
axes[0].set_xlabel("Day of Week")
axes[0].set_ylabel("Number of Transactions")

# barplot: Average bill amount
sns.barplot(data=tips, x="day", y="total_bill", errorbar="ci", palette="Set2", ax=axes[1])
axes[1].set_title("barplot(): Average Bill by Day (with CI)", fontweight="bold")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Average Total Bill ($)")

plt.tight_layout()
save_fig("../exports/08_categorical/barplot_vs_countplot.png")
plt.show()

print("✅ countplot shows frequency, barplot shows aggregated values")

## 3️⃣ Custom Estimators - Beyond Mean

The `estimator` parameter accepts any function that aggregates data. Common options:
- `np.mean` (default)
- `np.median`
- `np.sum`
- `np.std`
- Custom functions (e.g., 90th percentile, count, range)

In [ ]:
# 📊 barplot with custom estimators
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Mean (default)
sns.barplot(data=employee_df, x="Department", y="Salary", estimator=np.mean, 
            errorbar="ci", palette="Blues_d", ax=axes[0,0])
axes[0,0].set_title("Mean Salary by Department", fontweight="bold")
axes[0,0].set_ylabel("Mean Salary ($)")

# Median
sns.barplot(data=employee_df, x="Department", y="Salary", estimator=np.median, 
            errorbar=None, palette="Greens_d", ax=axes[0,1])
axes[0,1].set_title("Median Salary by Department", fontweight="bold")
axes[0,1].set_ylabel("Median Salary ($)")

# Sum (Total)
sns.barplot(data=employee_df, x="Department", y="Salary", estimator=np.sum, 
            errorbar=None, palette="Oranges_d", ax=axes[1,0])
axes[1,0].set_title("Total Salary by Department", fontweight="bold")
axes[1,0].set_ylabel("Total Salary ($)")

# Custom: 90th percentile
percentile_90 = lambda x: np.percentile(x, 90)
sns.barplot(data=employee_df, x="Department", y="Salary", estimator=percentile_90, 
            errorbar=None, palette="Purples_d", ax=axes[1,1])
axes[1,1].set_title("90th Percentile Salary by Department", fontweight="bold")
axes[1,1].set_ylabel("90th Percentile Salary ($)")

plt.tight_layout()
save_fig("../exports/08_categorical/custom_estimators.png")
plt.show()

print("✅ Custom estimators: mean, median, sum, and 90th percentile")

## 4️⃣ Grouped Barplots with Hue Parameter

Use `hue` to create side-by-side bars for comparing subcategories.

In [ ]:
# 📊 Grouped barplot with hue
plt.figure(figsize=(12, 5))
sns.barplot(
    data=tips,
    x="day",
    y="total_bill",
    hue="time",  # 👈 Group by time (Lunch vs Dinner)
    errorbar="sd",  # Standard deviation error bars
    palette="Set1",
    alpha=0.8
)
stylize_plot(
    title="Average Total Bill by Day and Time",
    xlabel="Day of Week",
    ylabel="Average Total Bill ($)"
)
plt.legend(title="Time", loc="upper right")
save_fig("../exports/08_categorical/grouped_barplot.png")
plt.show()

print("✅ Grouped barplot with hue for subcategory comparison")

## 🎯 Summary

You've now mastered advanced categorical plotting in Seaborn!

### **Key Takeaways:**

1. **barplot() - Aggregated Values:**
   - Shows central tendency (mean by default) with error bars
   - `estimator` parameter: `np.mean`, `np.median`, `np.sum`, custom functions
   - `errorbar` parameter: `"ci"`, `"sd"`, `"se"`, `("pi", 95)`, or `None`
   - Use for: comparing averages, totals, or medians across categories

2. **barplot() vs countplot():**
   - **countplot**: Frequency counts only (how many?)
   - **barplot**: Aggregated numeric values (what's the average/total/median?)

3. **Custom Estimators:**
   - Beyond mean: median, sum, std, percentiles
   - Create custom functions for specialized aggregations
   - Combine with error bars for uncertainty visualization

4. **Grouped Comparisons:**
   - Use `hue` parameter for side-by-side comparisons
   - Perfect for analyzing subcategories
   - Works with all categorical plot types

5. **Categorical Plot Comparison:**
   - **countplot/barplot**: Aggregated summaries
   - **boxplot/violinplot**: Full distribution with quartiles
   - **stripplot/swarmplot**: Individual data points
   - **pointplot**: Trends with error bars (better for trends over time/order)

### **Best Practices:**
- Use **barplot** when you want to show aggregated values with uncertainty
- Use **countplot** for simple frequency counts
- Choose **custom estimators** when mean doesn't tell the whole story
- Add **error bars** to show confidence or variability
- Use **hue** for multi-dimensional comparisons

### **Next Steps:**
- Learn advanced styling and color theory in Notebook 09
- Explore statistical parameters in depth in Notebook 10
- Practice combining multiple categorical plots in dashboards